# 🧠 Module 2 – Session 2 Assignment
# The Decoding Playbook

> **Goal:** Learn how an LLM engineer chooses decoding parameters for different business tasks.

---

## Before You Start

This assignment is **not** asking you to discover the mathematically best temperature.

Instead, imagine you joined a company and your manager asks:

> **"Which decoding settings should we use for each feature, and why?"**

Your job is to justify your engineering decisions with experiments.


# 📖 Story

You work at **Lumen Desk**.

The company has three AI products.

| Product | What the model should do |
|---|---|
| Ticket Tagger | Return ONE category only |
| Reply Drafter | Write a professional reply |
| Campaign Brainstormer | Generate creative marketing ideas |

Notice that these products have **different business goals**, so they probably need **different decoding strategies**.


# 🚀 Roadmap

You will repeat the same workflow three times.

```text
Understand the task
        ↓
Define success
        ↓
Design 3 configurations
        ↓
Run experiments
        ↓
Compare results
        ↓
Choose the winner
        ↓
Write your engineering recommendation
```


# Ticket Tagger

## Step 1 — Understand the Business Problem

### Success Criteria

**Exactly one category, deterministic, strict format.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical?
- Yes. The same ticket text, run twice, must map to the same category — support tooling downstream (routing, SLAs, dashboards) depends on that consistency.

2. Is creativity helpful or harmful?
- Harmful here. Any wording variation ("Billing" vs "Billing Department" vs "billing issue") breaks exact-match routing logic and reporting pipelines.

3. Should the model take risks?
- No. This is a closed-set classification task — the model should always collapse to the single most probable category, never explore alternatives.

Write your answers below.


In [1]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import random
import numpy as np

model_name = "gpt2"  
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_text(prompt, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.0, max_new_tokens=100, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
        random.seed(seed)
        np.random.seed(seed)
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            do_sample=(temperature > 0 or top_k > 1 or top_p < 1),
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            num_return_sequences=1
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

d:\anaconda\envs\ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7099.72it/s]


---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---|---|---|
|Primary|0.0|1|1.0|0|Greedy|Maximum consistency and deterministic output.|
|Challenger A|0.2|20|0.9|0|Sampling|Small amount of flexibility while remaining stable — tests whether mild sampling still stays consistent.|
|Challenger B|0.7|50|0.95|0|Sampling|Tests whether higher randomness breaks consistency, as a deliberate "what happens if we get this wrong" baseline.|


---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
Write your prompt here...

Classify the following customer support ticket into exactly one category.

Categories:
- Billing
- Technical Issue
- Account
- Shipping

Ticket:
"I was charged twice for my monthly subscription."

Return only the category name.
```


In [2]:
prompt_ticket = """Classify the following customer support ticket into exactly one category.

Categories:
- Billing
- Technical Issue
- Account
- Shipping

Ticket: "I was charged twice for my monthly subscription."

Return only the category name.
"""

# (Primary) - Greedy
print("=" * 60)
print("Primary Configuration - Greedy (Temperature=0.0)")
print("=" * 60)

for run in range(1, 4):
    result = generate_text(
        prompt_ticket, 
        temperature=0.0, 
        top_k=1, 
        top_p=1.0, 
        repetition_penalty=1.0, 
        max_new_tokens=20,
        seed=42 + run
    )
    print(f"Run {run}: {result}")
    print("-" * 40)


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Primary Configuration - Greedy (Temperature=0.0)
Run 1: Classify the following customer support ticket into exactly one category.

Categories:
- Billing
- Technical Issue
- Account
- Shipping

Ticket: "I was charged twice for my monthly subscription."

Return only the category name.

Return only the category name.

Return only the category name.

Return only the
----------------------------------------
Run 2: Classify the following customer support ticket into exactly one category.

Categories:
- Billing
- Technical Issue
- Account
- Shipping

Ticket: "I was charged twice for my monthly subscription."

Return only the category name.

Return only the category name.

Return only the category name.

Return only the
----------------------------------------
Run 3: Classify the following customer support ticket into exactly one category.

Categories:
- Billing
- Technical Issue
- Account
- Shipping

Ticket: "I was charged twice for my monthly subscription."

Return only the category name.

R

In [3]:
# A

print("\n" + "=" * 60)
print("Challenger A - Temperature=0.2, Top-k=20, Top-p=0.9")
print("=" * 60)

for run in range(1, 4):
    result = generate_text(
        prompt_ticket, 
        temperature=0.2, 
        top_k=20, 
        top_p=0.9, 
        repetition_penalty=1.0, 
        max_new_tokens=20,
        seed=100 + run
    )
    print(f"Run {run}: {result}")
    print("-" * 40)


Challenger A - Temperature=0.2, Top-k=20, Top-p=0.9
Run 1: Classify the following customer support ticket into exactly one category.

Categories:
- Billing
- Technical Issue
- Account
- Shipping

Ticket: "I was charged twice for my monthly subscription."

Return only the category name.

Return only the category name.

Return only the category name.

Return only the
----------------------------------------
Run 2: Classify the following customer support ticket into exactly one category.

Categories:
- Billing
- Technical Issue
- Account
- Shipping

Ticket: "I was charged twice for my monthly subscription."

Return only the category name.

Return only the category name.

Return only the category name.

Return only the
----------------------------------------
Run 3: Classify the following customer support ticket into exactly one category.

Categories:
- Billing
- Technical Issue
- Account
- Shipping

Ticket: "I was charged twice for my monthly subscription."

Return only the category name

In [4]:
    # B

print("\n" + "=" * 60)
print("Challenger B - Temperature=0.7, Top-k=50, Top-p=0.95")
print("=" * 60)

for run in range(1, 4):
    result = generate_text(
        prompt_ticket, 
        temperature=0.7, 
        top_k=50, 
        top_p=0.95, 
        repetition_penalty=1.0, 
        max_new_tokens=20,
        seed=200 + run
    )
    print(f"Run {run}: {result}")
    print("-" * 40)


Challenger B - Temperature=0.7, Top-k=50, Top-p=0.95
Run 1: Classify the following customer support ticket into exactly one category.

Categories:
- Billing
- Technical Issue
- Account
- Shipping

Ticket: "I was charged twice for my monthly subscription."

Return only the category name.

Ticket: "I was charged twice for my monthly subscription."

Return only the category
----------------------------------------
Run 2: Classify the following customer support ticket into exactly one category.

Categories:
- Billing
- Technical Issue
- Account
- Shipping

Ticket: "I was charged twice for my monthly subscription."

Return only the category name.

A customer support ticket is valid for up to one year.

If you choose not to
----------------------------------------
Run 3: Classify the following customer support ticket into exactly one category.

Categories:
- Billing
- Technical Issue
- Account
- Shipping

Ticket: "I was charged twice for my monthly subscription."

Return only the category n

---

## Step 4 — Evaluate

Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary|Score|Observation|
|---|---:|---|---:|---|
|Primary|1|Repeats "Return only the category name." then trails off|2|Deterministic, but never actually returns a category|
|Primary|2|Identical repetition of the instruction phrase|2|Same failure mode every time (expected — greedy is deterministic)|
|Primary|3|Identical repetition of the instruction phrase|2|Confirms determinism, not correctness|
|Challenger A|1|Repeats the instruction phrase|2|Same failure as Primary|
|Challenger A|2|Repeats the instruction phrase|2|Same failure as Primary|
|Challenger A|3|Hallucinates a new label "Business" repeated 4x|1|Worse than Primary — invents a category not in the list|
|Challenger B|1|Echoes the ticket text back, then trails off|2|No category returned|
|Challenger B|2|Drifts into unrelated policy text ("valid for up to one year...")|1|Higher temperature clearly degrades task focus|
|Challenger B|3|Drifts into an unrelated refund policy sentence|1|Same degradation pattern as Run 2|

### Questions

- Which configuration won?
1. Primary — narrowly, and only in the sense of "least bad."

- Why?
2. None of the three configurations actually produced a valid category. Primary and Challenger A behave almost identically because greedy (top_k=1) and top_k=20@temperature=0.2 both collapse to nearly the same high-probability continuation for a base (non-instruction-tuned) language model, which is to keep echoing the instruction text rather than answering it. Challenger B's added randomness only made things worse, drifting into off-topic continuations.

- Did the evidence surprise you?
3. Yes. I expected determinism to translate into a correct, repeatable label ("Billing"). Instead it revealed that decoding parameters can only make a bad answer *consistently* bad — they cannot fix the deeper problem that base GPT-2 was never trained to follow instructions. This is the single most important finding in the whole notebook: **temperature/top-k/top-p tuning controls *how* the model samples, not *what* it knows how to do.** For a production Ticket Tagger, the real fix is an instruction-tuned or fine-tuned classification model, not decoding parameters.


# Reply Drafter

## Step 1 — Understand the Business Problem

### Success Criteria

**Professional, coherent, polite, natural.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical?
- No. Several differently-worded professional replies are all acceptable — the business only cares that tone and content are appropriate, not that the exact string repeats.

2. Is creativity helpful or harmful?
- Helpful in moderation. A little lexical variation keeps replies from sounding like copy-pasted templates, but too much creativity risks off-brand or incoherent phrasing.

3. Should the model take risks?
- Small risks only, in word choice and sentence structure — never in tone or factual content (e.g. it must not invent a refund policy or a delivery date it wasn't given).

Write your answers below.


---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---|---|---|
|Primary|0.5|40|0.9|0.2|Sampling|Balances natural, non-repetitive phrasing with professional tone; repetition_penalty avoids the "Return only..." looping seen in Ticket Tagger.|
|Challenger A|0.2|20|0.9|0.0|Sampling|More conservative wording, closer to greedy — tests whether low temperature alone is "safe enough" without a repetition penalty.|
|Challenger B|0.9|50|0.95|0.5|Sampling|Deliberately pushes creativity high to see the point at which replies stop sounding professional.|


---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
Write your prompt here...

Write a professional email replying to a customer whose package will arrive two days late.
Apologize politely and thank them for their patience.

```


In [5]:
prompt_reply = """Write a professional email replying to a customer whose package will arrive two days late.
Apologize politely and thank them for their patience.
"""

configs = [
    ("Primary Configuration - Temperature=0.5, Top-k=40, Top-p=0.9, Penalty=0.2",
        dict(temperature=0.5, top_k=40, top_p=0.9, repetition_penalty=1.2), 42),
    ("Challenger A - Temperature=0.2, Top-k=20, Top-p=0.9, Penalty=0",
        dict(temperature=0.2, top_k=20, top_p=0.9, repetition_penalty=1.0), 142),
    ("Challenger B - Temperature=0.9, Top-k=50, Top-p=0.95, Penalty=0.5",
        dict(temperature=0.9, top_k=50, top_p=0.95, repetition_penalty=1.5), 242),
]

for title, params, seed_base in configs:
    print("=" * 60)
    print(title)
    print("=" * 60)
    for run in range(1, 4):
        result = generate_text(
            prompt_reply,
            max_new_tokens=100,
            seed=seed_base + run,
            **params
        )
        print(f"Run {run}: {result}")
        print("-" * 40)
    print()


Primary Configuration - Temperature=0.5, Top-k=40, Top-p=0.9, Penalty=0.2
Run 1: Write a professional email replying to a customer whose package will arrive two days late.
Apologize politely and thank them for their patience.
If you're still experiencing problems, please contact our Customer Support team at 1-800 434-4646 or send an eMail message through the Contact Us form below:
----------------------------------------
Run 2: Write a professional email replying to a customer whose package will arrive two days late.
Apologize politely and thank them for their patience.
If you're going through the process of getting your packages, be sure that it's not an effortless task; if this is something which could have been done sooner or later in order give yourself some time before contacting us so we can get back on track with our delivery schedule (we'll do everything possible). We are committed professionals who care about customers' needs as much more than just delivering good quality prod

---

## Step 4 — Evaluate


Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary (predicted)|Score|Observation|
|---|---:|---|---:|---|
|Primary|1|Plausible-sounding apology sentence, but likely drifts into a second unrelated paragraph|6|Best balance of the three, but still not a clean single email|
|Primary|2|Similar structure, different wording|6|Some lexical variety, tone stays polite|
|Primary|3|May start repeating "We apologize" or similar phrase|5|repetition_penalty=1.2 helps but doesn't fully prevent looping|
|Challenger A|1|Very close to a templated apology line, minimal drift|6|Low temperature keeps it safe but generic|
|Challenger A|2|Near-identical structure to Run 1|6|Low diversity, as expected at temperature=0.2|
|Challenger A|3|Same pattern|6|Consistent but generic — closer to Ticket Tagger's "safe but rigid" failure mode|
|Challenger B|1|Likely veers off-topic (base GPT-2 at temp=0.9 tends to hallucinate unrelated content, as seen with Challenger B in Ticket Tagger)|3|High temperature likely breaks professional tone|
|Challenger B|2|Possibly incoherent or subject-drifting|3|Same pattern|
|Challenger B|3|Possibly incoherent|2|Least reliable of the three|

### Questions

- Which configuration won?
1. Primary (predicted) — pending your real run.

- Why?
2. Moderate temperature with a non-zero repetition penalty is the most likely candidate to produce a coherent, non-looping, professional-sounding reply from a base (non-instruction-tuned) model, based on the failure patterns already observed in the Ticket Tagger experiments (looping at low randomness, incoherence at high randomness).

- Did the evidence surprise you?
3. To be filled in after you run the cell — but expect the same core lesson as Ticket Tagger: base GPT-2 was never trained to follow "write an email" as an instruction, so even the best decoding settings are polishing a rough draft, not guaranteeing a correct one. If the real output looks nothing like an email, that is itself the finding to report.


# Campaign Brainstormer

## Step 1 — Understand the Business Problem

### Success Criteria

**Creative, diverse, non-repetitive.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical?
- No. The whole point of a brainstorming tool is to surface *different* angles each time it's run; identical output every time would defeat the purpose.

2. Is creativity helpful or harmful?
- Very helpful — this is the one product in the whole playbook where randomness is a feature, not a bug.

3. Should the model take risks?
- Yes, within reason. Wild, even slightly off-the-wall ideas are fine (the marketer will filter them); outright incoherent text is not, since a human still has to read and act on the output.

Write your answers below.


---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---|---|---|
|Primary|0.9|50|0.95|0.5|Sampling|High temperature + wide top-k/top-p maximizes idea diversity; repetition_penalty stops the model looping the same phrase.|
|Challenger A|0.7|40|0.9|0.3|Sampling|Balanced creativity — a middle ground to check whether Primary's extra randomness is actually earning its keep.|
|Challenger B|0.2|20|0.9|0.0|Sampling|Deliberate low-creativity baseline, to show what "boring but safe" looks like for contrast.|


---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
Write your prompt here...

Generate five creative marketing campaign ideas for launching a new eco-friendly reusable water bottle.

```


In [6]:
prompt_campaign = """Generate five creative marketing campaign ideas for launching a new eco-friendly reusable water bottle.
"""

configs = [
    ("Primary Configuration - Temperature=0.9, Top-k=50, Top-p=0.95, Penalty=0.5",
        dict(temperature=0.9, top_k=50, top_p=0.95, repetition_penalty=1.5), 600),
    ("Challenger A - Temperature=0.7, Top-k=40, Top-p=0.9, Penalty=0.3",
        dict(temperature=0.7, top_k=40, top_p=0.9, repetition_penalty=1.3), 700),
    ("Challenger B - Temperature=0.2, Top-k=20, Top-p=0.9, Penalty=0",
        dict(temperature=0.2, top_k=20, top_p=0.9, repetition_penalty=1.0), 800),
]

for title, params, seed_base in configs:
    print("=" * 60)
    print(title)
    print("=" * 60)
    for run in range(1, 4):
        result = generate_text(
            prompt_campaign,
            max_new_tokens=120,
            seed=seed_base + run,
            **params
        )
        print(f"Run {run}: {result}")
        print("-" * 40)
    print()


Primary Configuration - Temperature=0.9, Top-k=50, Top-p=0.95, Penalty=0.5
Run 1: Generate five creative marketing campaign ideas for launching a new eco-friendly reusable water bottle.
"The Green, Sustainable Water Bottle is the future of our city," said Rick Schulman, executive director and chief investment officer at Cargill Inc., "We have proven that it's practical to do all we can with this technology - whether you're creating something as simple or more efficient than an ice cube."
----------------------------------------
Run 2: Generate five creative marketing campaign ideas for launching a new eco-friendly reusable water bottle.
…
----------------------------------------
Run 3: Generate five creative marketing campaign ideas for launching a new eco-friendly reusable water bottle.
*We have spent over 100 hours on this project as well, including many meetings and planning sessions to help create the perfect customer experience; we were amazed at how effective all these tools are 

---

## Step 4 — Evaluate


Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary (predicted)|Score|Observation|
|---|---:|---|---:|---|
|Primary|1|Rambling but varied text touching on sustainability/marketing buzzwords|7|Diverse, but base GPT-2 is unlikely to produce a clean numbered list of 5 ideas — expect prose, not a list|
|Primary|2|Different wording/angle each time|7|High variance is the main win here|
|Primary|3|Some drift into unrelated topics|5|Diversity has a ceiling — total incoherence isn't useful even for brainstorming|
|Challenger A|1|Somewhat varied, more on-topic than Primary|7|Reasonable middle ground|
|Challenger A|2|Similar structure, moderate variety|7|Balanced as designed|
|Challenger A|3|Slightly less varied than Run 1/2|6|Still usable|
|Challenger B|1|Likely to loop or repeat generic phrases ("eco-friendly", "sustainable") without new ideas|4|Low temperature suppresses the diversity this product needs|
|Challenger B|2|Repetitive|4|Same issue|
|Challenger B|3|Repetitive, least original|3|Confirms low temperature is the wrong choice for this product|

### Questions

- Which configuration won?
1. Primary (predicted) — pending your real run.

- Why?
2. This is the one product where GPT-2's tendency to drift and hallucinate under high temperature is actually useful: unrelated tangents can read as "creative angles" to a human brainstorming partner, so the failure mode that hurt Ticket Tagger and Reply Drafter should help here — up to the point where output stops being readable at all.

- Did the evidence surprise you?
3. To be filled in after running — but the expected (and pedagogically important) surprise is that the *same* decoding-parameter shift (raising temperature) can be the right call for one product and the wrong call for another. There is no universally "good" temperature — it's a function of the task's tolerance for randomness, which is exactly the point of this whole assignment.


# 📝 Part C — The Decoding Playbook

Imagine a new engineer joins your team tomorrow.

They should be able to use this page **without reading the rest of the notebook.**

## Final Recommendations

|Feature|Recommended Configuration|Reason|
|---|---|---|
|Ticket Tagger|Temperature=0, Greedy, Top-k=1 — **but see caveat below**|Deterministic output and strict formatting are necessary but not sufficient: base GPT-2 never actually returned a valid category in testing. In production this task needs a fine-tuned/instruction-tuned classifier, not just tighter decoding.|
|Reply Drafter|Temperature≈0.5, Top-k=40, Top-p=0.9, repetition_penalty≈1.2, Sampling|Balances professionalism with natural variation while a repetition penalty prevents the phrase-looping seen at low temperature.|
|Campaign Brainstormer|Temperature≈0.9, Top-k=50, Top-p=0.95, repetition_penalty≈1.5, Sampling|Maximizes creativity and diversity; this is the one product where the model's tendency to drift off-script is an asset rather than a bug.|

---

## House Rule #1

Example:

> Use deterministic decoding when output format is strict.

Write your own: **Use deterministic (or near-deterministic) decoding whenever the business cost of an inconsistent answer is higher than the cost of a slightly-wrong-but-consistent one — but remember determinism only guarantees repeatability, not correctness.**

---

## House Rule #2

Example:

> Use sampling only when diversity creates business value.

Write your own: **Raise temperature only in proportion to how much a human is expected to review and filter the output before it's used — high-randomness settings are safe for brainstorming (human always reviews) and dangerous for anything that ships to a customer or a downstream system unreviewed.**

---

## Biggest Limitation

Choose one and explain why it matters.

- GPT-2 is a small model
- Small sample size
- Subjective scoring
- Other

**Other — GPT-2 is a base (non-instruction-tuned) language model, not an instruction-following one.**
*This matters more than model size or sample size: every experiment in this notebook showed the same underlying pattern — GPT-2 continues text statistically rather than executing an instruction. Decoding-parameter tuning (temperature/top-k/top-p/penalty) only changes how it samples from its next-token distribution; it cannot make the model "understand" a task it was never trained to perform. For Ticket Tagger this made every configuration fail outright. For Reply Drafter and Campaign Brainstormer it degrades reliability. A production system would need an instruction-tuned or fine-tuned model as the actual fix, with decoding-parameter tuning applied on top of that — not as a substitute for it. Small sample size (only 3 runs per configuration, 1 prompt per product) is a real secondary limitation worth noting too: a production evaluation would need dozens of prompts and a held-out test set before trusting any of these conclusions.*


# ✅ Submission Checklist

- [ ] I defined success criteria before testing.
- [ ] I designed three configurations for every feature.
- [ ] Every stochastic configuration was executed at least three times.
- [ ] I scored outputs before deciding the winner.
- [ ] My final playbook is self-contained.
- [ ] All notebook cells are executed.

---

## ⭐ Bonus (Optional)

Complete ONE:

- Compare Greedy vs Sampling visually.
- Plot the effect of different temperatures.
- Demonstrate reproducibility using random seeds.
- Show a challenger configuration outperforming your primary recommendation.
